In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    USER_DAY_FILE,
    USER_BASELINE_FILE,
    GROUND_TRUTH_PARQUET,
)


# ============================================================
# HELPERS
# ============================================================

def db_path(path):
    return Path(path).resolve().as_posix()


# ============================================================
# INPUT CHECK
# ============================================================

required_files = {
    "User-Day dataset": USER_DAY_FILE,
    "User baseline": USER_BASELINE_FILE,
    "Ground truth": GROUND_TRUTH_PARQUET,
}

missing = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not Path(path).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required validation files:\n"
        + "\n".join(missing)
    )

print("✓ Required validation files found.")


# ============================================================
# PATHS
# ============================================================

user_day = db_path(USER_DAY_FILE)
baseline = db_path(USER_BASELINE_FILE)
ground_truth = db_path(GROUND_TRUTH_PARQUET)


# ============================================================
# START VALIDATION
# ============================================================

con = duckdb.connect()

print("\n" + "=" * 70)
print("BASELINE & DATA VALIDATION")
print("=" * 70)


# ============================================================
# 1. FIRST CERT R4.2 MALICIOUS INCIDENT
# ============================================================

first_attack = con.execute(f"""
    SELECT
        MIN(attack_start) AS first_attack

    FROM read_parquet(
        '{ground_truth}'
    )
""").df()

first_attack_value = (
    first_attack.iloc[0]["first_attack"]
)

baseline_cutoff = (
    pd.Timestamp(first_attack_value)
    .date()
)

print("\n[1/7] First malicious incident:")
display(first_attack)

print(
    f"Baseline cutoff date : "
    f"{baseline_cutoff}"
)


# ============================================================
# 2. USER-DAY STRUCTURE
# ============================================================

summary = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT user) AS users,
        MIN(day) AS first_day,
        MAX(day) AS last_day

    FROM read_parquet(
        '{user_day}'
    )
""").df()

print("\n[2/7] User-Day summary:")
display(summary)


# ============================================================
# 3. EXTERNAL EMAIL SANITY CHECK
# ============================================================

email_check = con.execute(f"""
    SELECT
        SUM(
            email_count
        ) AS total_emails,

        SUM(
            external_email_events
        ) AS external_email_events,

        ROUND(
            100.0
            * SUM(external_email_events)
            / NULLIF(
                SUM(email_count),
                0
            ),
            2
        ) AS external_percentage,

        SUM(
            CASE
                WHEN external_email_events > email_count
                THEN 1
                ELSE 0
            END
        ) AS invalid_rows

    FROM read_parquet(
        '{user_day}'
    )
""").df()

print("\n[3/7] External email sanity check:")
display(email_check)


# ============================================================
# 4. AAF0535 BASELINE-PERIOD EMAIL CHECK
# ============================================================

aaf_email = con.execute(f"""
    SELECT
        COUNT(*) AS baseline_days,

        SUM(
            email_count
        ) AS emails,

        SUM(
            external_email_events
        ) AS external_events

    FROM read_parquet(
        '{user_day}'
    )

    WHERE user = 'AAF0535'
      AND day < DATE '{baseline_cutoff}'
      AND is_weekend = 0
      AND logon_count > 0
""").df()

print("\n[4/7] AAF0535 baseline-period email activity:")
display(aaf_email)


# ============================================================
# 5. BASELINE SUMMARY
# ============================================================

baseline_summary = con.execute(f"""
    SELECT
        COUNT(*) AS users,
        MIN(
            baseline_workdays
        ) AS min_days,

        MEDIAN(
            baseline_workdays
        ) AS median_days,

        MAX(
            baseline_workdays
        ) AS max_days

    FROM read_parquet(
        '{baseline}'
    )
""").df()

print("\n[5/7] Baseline summary:")
display(baseline_summary)


# ============================================================
# 6. AAF0535 SAVED BASELINE
# ============================================================

aaf_baseline = con.execute(f"""
    SELECT
        user,
        baseline_workdays,

        email_count_median,
        email_count_p95,

        external_email_median,
        external_email_p95,

        usb_connect_median,
        usb_connect_p95,

        usb_files_median,
        usb_files_p95,

        http_requests_median,
        http_requests_p95,

        unique_domains_median,
        unique_domains_p95

    FROM read_parquet(
        '{baseline}'
    )

    WHERE user = 'AAF0535'
""").df()

print("\n[6/7] AAF0535 saved baseline:")
display(aaf_baseline)


# ============================================================
# 7. DUPLICATE CHECKS
# ============================================================

user_day_duplicates = con.execute(f"""
    SELECT COUNT(*)

    FROM (
        SELECT
            user,
            day,
            COUNT(*) AS n

        FROM read_parquet(
            '{user_day}'
        )

        GROUP BY
            user,
            day

        HAVING COUNT(*) > 1
    )
""").fetchone()[0]


baseline_duplicates = con.execute(f"""
    SELECT COUNT(*)

    FROM (
        SELECT
            user,
            COUNT(*) AS n

        FROM read_parquet(
            '{baseline}'
        )

        GROUP BY user

        HAVING COUNT(*) > 1
    )
""").fetchone()[0]


print("\n[7/7] Duplicate checks:")

print(
    f"Duplicate User-Day groups : "
    f"{user_day_duplicates}"
)

print(
    f"Duplicate baseline users  : "
    f"{baseline_duplicates}"
)


# ============================================================
# AUTOMATED VALIDATION
# ============================================================

rows = int(
    summary.iloc[0]["rows"]
)

users = int(
    summary.iloc[0]["users"]
)

external_pct = float(
    email_check.iloc[0][
        "external_percentage"
    ]
)

invalid_email_rows = int(
    email_check.iloc[0][
        "invalid_rows"
    ]
)

baseline_users = int(
    baseline_summary.iloc[0][
        "users"
    ]
)

min_baseline_days = int(
    baseline_summary.iloc[0][
        "min_days"
    ]
)


checks = {
    "First attack is 2010-06-10":
        str(first_attack_value).startswith(
            "2010-06-10"
        ),

    "User-Day has 464,687 rows":
        rows == 464687,

    "User-Day has 1,000 users":
        users == 1000,

    "External email percentage is 48.06%":
        abs(
            external_pct - 48.06
        ) < 0.01,

    "No external-email logical errors":
        invalid_email_rows == 0,

    "Baseline contains 1,000 users":
        baseline_users == 1000,

    "Minimum baseline history >= 20 workdays":
        min_baseline_days >= 20,

    "AAF0535 exists exactly once in baseline":
        len(aaf_baseline) == 1,

    "No duplicate User-Day records":
        user_day_duplicates == 0,

    "No duplicate baseline users":
        baseline_duplicates == 0,
}


# ============================================================
# VALIDATION RESULTS
# ============================================================

validation_results = pd.DataFrame(
    [
        {
            "Check": name,
            "Status": (
                "PASS"
                if passed
                else "FAIL"
            ),
        }
        for name, passed in checks.items()
    ]
)

print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

display(validation_results)


all_passed = all(
    checks.values()
)


print("=" * 70)

if all_passed:
    print(
        "✅ DATA & BASELINE VALIDATION: PASS"
    )

else:
    print(
        "❌ DATA & BASELINE VALIDATION: "
        "REVIEW REQUIRED"
    )

print("=" * 70)

✓ Required validation files found.

BASELINE & DATA VALIDATION

[1/7] First malicious incident:


,first_attack
0,2010-06-10 07:54:10


Baseline cutoff date : 2010-06-10

[2/7] User-Day summary:


,rows,users,first_day,last_day
0,464687,1000,2010-01-02,2011-05-17



[3/7] External email sanity check:


,total_emails,external_email_events,external_percentage,invalid_rows
0,2629979.0,1264049.0,48.06,0.0


... (5 outputs left out)

,user,baseline_workdays,email_count_median,email_count_p95,external_email_median,external_email_p95,usb_connect_median,usb_connect_p95,usb_files_median,usb_files_p95,http_requests_median,http_requests_p95,unique_domains_median,unique_domains_p95
0,AAF0535,111,3.0,3.0,1.0,3.0,2.0,3.0,2.0,5.0,29.0,29.0,16.0,20.5



[7/7] Duplicate checks:
Duplicate User-Day groups : 0
Duplicate baseline users  : 0

VALIDATION RESULTS


,Check,Status
0,First attack is 2010-06-10,PASS
1,"User-Day has 464,687 rows",PASS
2,"User-Day has 1,000 users",PASS
3,External email percentage is 48.06%,PASS
4,No external-email logical errors,PASS
5,"Baseline contains 1,000 users",PASS
6,Minimum baseline history >= 20 workdays,PASS
7,AAF0535 exists exactly once in baseline,PASS
8,No duplicate User-Day records,PASS
9,No duplicate baseline users,PASS


✅ DATA & BASELINE VALIDATION: PASS


In [3]:
# ============================================================
# BASELINE CONSISTENCY CHECK
# ============================================================

print("=" * 70)
print("BASELINE CONSISTENCY CHECK")
print("=" * 70)


# ============================================================
# CALCULATE DIRECTLY FROM CURRENT USER-DAY DATA
# ============================================================

calculated = con.execute(f"""
    SELECT
        COUNT(*) AS baseline_workdays,

        SUM(
            external_email_events
        ) AS external_sum,

        MEDIAN(
            external_email_events
        ) AS external_email_median,

        QUANTILE_CONT(
            external_email_events,
            0.95
        ) AS external_email_p95,

        MEDIAN(
            files_copied_to_usb
        ) AS usb_files_median,

        QUANTILE_CONT(
            files_copied_to_usb,
            0.95
        ) AS usb_files_p95

    FROM read_parquet(
        '{user_day}'
    )

    WHERE user = 'AAF0535'
      AND day < DATE '{baseline_cutoff}'
      AND is_weekend = 0
      AND logon_count > 0
""").df()


# ============================================================
# READ SAVED BASELINE VALUES
# ============================================================

saved = con.execute(f"""
    SELECT
        baseline_workdays,

        external_email_median,
        external_email_p95,

        usb_files_median,
        usb_files_p95

    FROM read_parquet(
        '{baseline}'
    )

    WHERE user = 'AAF0535'
""").df()


print(
    "\nCalculated directly "
    "from current User-Day data:"
)

display(calculated)


print(
    "\nSaved in baseline file:"
)

display(saved)


# ============================================================
# COMPARE VALUES
# ============================================================

consistency_checks = {
    "Baseline workdays match":
        int(
            calculated.iloc[0][
                "baseline_workdays"
            ]
        )
        ==
        int(
            saved.iloc[0][
                "baseline_workdays"
            ]
        ),

    "External email median matches":
        float(
            calculated.iloc[0][
                "external_email_median"
            ]
        )
        ==
        float(
            saved.iloc[0][
                "external_email_median"
            ]
        ),

    "External email P95 matches":
        abs(
            float(
                calculated.iloc[0][
                    "external_email_p95"
                ]
            )
            -
            float(
                saved.iloc[0][
                    "external_email_p95"
                ]
            )
        ) < 0.0001,

    "USB files median matches":
        float(
            calculated.iloc[0][
                "usb_files_median"
            ]
        )
        ==
        float(
            saved.iloc[0][
                "usb_files_median"
            ]
        ),

    "USB files P95 matches":
        abs(
            float(
                calculated.iloc[0][
                    "usb_files_p95"
                ]
            )
            -
            float(
                saved.iloc[0][
                    "usb_files_p95"
                ]
            )
        ) < 0.0001,
}


consistency_results = pd.DataFrame(
    [
        {
            "Check": name,
            "Status": (
                "PASS"
                if passed
                else "FAIL"
            ),
        }
        for name, passed
        in consistency_checks.items()
    ]
)


print("\nConsistency results:")

display(consistency_results)


consistency_passed = all(
    consistency_checks.values()
)


print("=" * 70)

if consistency_passed:
    print(
        "✅ BASELINE CONSISTENCY: PASS"
    )

else:
    print(
        "❌ BASELINE CONSISTENCY: "
        "REVIEW REQUIRED"
    )

print("=" * 70)


# ============================================================
# FINAL NOTEBOOK STATUS
# ============================================================

if all_passed and consistency_passed:
    print(
        "\n✅ BASELINE & DATA VALIDATION "
        "NOTEBOOK: PASS"
    )


con.close()

BASELINE CONSISTENCY CHECK

Calculated directly from current User-Day data:


,baseline_workdays,external_sum,external_email_median,external_email_p95,usb_files_median,usb_files_p95
0,111,118.0,1.0,3.0,2.0,5.0



Saved in baseline file:


,baseline_workdays,external_email_median,external_email_p95,usb_files_median,usb_files_p95
0,111,1.0,3.0,2.0,5.0



Consistency results:


,Check,Status
0,Baseline workdays match,PASS
1,External email median matches,PASS
2,External email P95 matches,PASS
3,USB files median matches,PASS
4,USB files P95 matches,PASS


✅ BASELINE CONSISTENCY: PASS

✅ BASELINE & DATA VALIDATION NOTEBOOK: PASS
